In [1]:
import os
import json
import base64
import numpy as np
import pandas as pd
import geopandas as gpd
from typing import Optional
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThanOrEqual
from pyiceberg.catalog import load_catalog

In [2]:
us_cbsa = gpd.read_file("/Users/houpuli/Downloads/Core_based_statistical_area_for_the_US_July_2023_4764927935501855778.geojson")
us_msa = us_cbsa[us_cbsa['CBSATYPE'] == 'Metropolitan Statistical Area']
msa_bspo = us_msa[us_msa['CBSACODE']=='14740']
msa_bspo

,OBJECTID,CBSACODE,CBSANAME,CBSATYPE,ALAND,AWATER,geometry
113,114,14740,"Bremerton-Silverdale-Port Orchard, WA",Metropolitan Statistical Area,1023308789,442187072,"POLYGON ((-122.45480 47.57619, -122.45486 47.5..."


In [ ]:
import pandas as pd
from glob import glob

folder = "/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw"
files = glob(folder + "/*.csv.gz")

# print(len(files))  

dfs = []

for f in files:
    print("reading:", f)
    dfs.append(pd.read_csv(f))

bsop_sf = pd.concat(dfs, ignore_index=True)
bsop_sf = bsop_sf[['PLACEKEY','PARENT_PLACEKEY','LOCATION_NAME', 'TRACKING_CLOSED_SINCE', 'LATITUDE','LONGITUDE','TOP_CATEGORY','SUB_CATEGORY','CATEGORY_TAGS','NAICS_CODE','STREET_ADDRESS']]
bsop_sf = gpd.GeoDataFrame(bsop_sf, geometry=gpd.points_from_xy(bsop_sf["LONGITUDE"], bsop_sf["LATITUDE"]), crs="EPSG:4326")
bsop_sf = bsop_sf.drop(columns=['LATITUDE','LONGITUDE'])

msa_bspo = msa_bspo.to_crs(bsop_sf.crs)
bsop_sf = gpd.sjoin(bsop_sf, msa_bspo, how='inner',predicate="within").reset_index(drop=True)
bsop_sf = bsop_sf.drop(columns=['index_right','OBJECTID','CBSACODE','CBSANAME','CBSATYPE','ALAND','AWATER'])
# bsop_sf.to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/bsop_sf.geojson', driver='GeoJSON')

reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_0_4_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_0_5_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_1_6_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_1_7_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_2_2_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_2_3_0.csv.gz
reading: /Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/BSOP_MSA/bspo msa/safegraph_raw/global-places-poi-geometry_3_0_0.csv.gz
reading: /Users/houp

In [3]:
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree

def search_spatial_candidates(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    k: int = 100,
    max_dist: float = 1000, 
    id_col: str = "id",
    crs_for_distance: int = 3857,
):
    """
    Attach k nearest compared POI ids & distances to reference_gdf.

    Returns
    -------
    GeoDataFrame with two new columns:
    - cand_ids   : list of compared ids
    - cand_dist_m: list of distances (meters)
    """

    ref_proj = reference_gdf.to_crs(crs_for_distance)
    cmp_proj = compared_gdf.to_crs(crs_for_distance)

    ref_xy = np.column_stack([ref_proj.geometry.x, ref_proj.geometry.y])
    cmp_xy = np.column_stack([cmp_proj.geometry.x, cmp_proj.geometry.y])

    tree = cKDTree(cmp_xy)
    k_eff = min(k, len(compared_gdf))

    dist, idx = tree.query(ref_xy, k=k_eff)

    if k_eff == 1:
        dist = dist.reshape(-1, 1)
        idx = idx.reshape(-1, 1)

    cmp_ids = compared_gdf[id_col].to_numpy()

    cand_ids = []
    cand_dists = []

    for row_idx, row_dist in zip(idx, dist):
        ids = []
        dists = []

        for j, d in zip(row_idx, row_dist):
            if d <= max_dist:
                ids.append(cmp_ids[j])
                dists.append(d)

        cand_ids.append(ids)
        cand_dists.append(dists)

    result = reference_gdf.copy()
    result["cand_ids"] = cand_ids
    result["cand_dist_m"] = cand_dists

    return result

In [4]:
FOOD_WORDS = {
    "RESTAURANT","RESTURANT","RESTARAUNT",
    "CAFE","CAFÉ","COFFEE","BAR","BISTRO","GRILL",
    "KITCHEN","DINER","EATERY","STEAKHOUSE",
    "SEAFOOD","BUFFET","BBQ","PIZZA","SUSHI","RAMEN",
    "NOODLE","NOODLES","BURGER","BURGERS","TACO","TACOS",
    "CHICKEN","WINGS","BAKERY","DELI","DELICATESSEN",
    "COURT","FOOD","EXPRESS","HOUSE","SHOP"
}

PLACE_WORDS = {
    "HALL","CENTER","CENTRE","PLAZA","MARKET","MALL",
    "GARDEN","GARDENS","PARK","SQUARE","TOWER","TOWERS",
    "STATION","TERMINAL","BUILDING","GALLERY","THEATER","SCHOOL","COURT","INN",
    "HOTEL","MOTEL","INN","SUITES","SUITE",
    "SPA","SALON","STUDIO","CENTER","CENTRE",
    "CLUB","LOUNGE","STATION","STOP"
}

LEGAL_WORDS = {
    "LLC","INC","CORP","CORPORATION","CO","COMPANY",
    "LTD","LIMITED","GROUP","HOLDINGS","OFFICE"
}

GRAMMAR = {
    "THE","OF","AT","AND","FOR","IN","ON","BY","WITH","TO","FROM"
}

NON_PRIMARY_TOKENS = FOOD_WORDS | PLACE_WORDS | LEGAL_WORDS | GRAMMAR

In [5]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import unicodedata


def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = re.sub(r"\b'S\b", "", s) # new change
    s = re.sub(r"\bS\b", "", s) # new change
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def extract_prinmary_str(name):

    tokens = name.split()
    core = [t for t in tokens if t not in NON_PRIMARY_TOKENS]
    if len(core) == 1 and len(core[0]) < 3:
        return name
    if core:
        return " ".join(core)
    return name

def match_by_name(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    re_name_col: str = "name",
    comp_name_col: str = "name",
    comp_id: str = "id",
    comp_id_col: str="cat_main",
    threshold: int = 80,
):
    """
    Perform WRatio name matching within spatial candidates.

    Returns
    -------
    GeoDataFrame with:
    - matched_id_name
    - name_score
    """

    # clean names for matching
    id_to_name_clean = compared_gdf.set_index(comp_id)[comp_name_col].apply(clean_name).apply(extract_prinmary_str).to_dict()
    # raw names for storage
    id_to_name_raw = compared_gdf.set_index(comp_id)[comp_name_col].to_dict()
    # raw compared df category
    id_to_cat = compared_gdf.set_index(comp_id)[comp_id_col].to_dict()

    matched_ids = []
    scores = []
    loc_dists = []
    matched_names = []
    matched_cats = []

    for _, row in reference_gdf.iterrows():
        query = extract_prinmary_str(clean_name(row.get(re_name_col)))

        if not isinstance(query, str) or not row["cand_ids"]:
            matched_ids.append(pd.NA)
            scores.append(pd.NA)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)
            continue

        cand_names = [id_to_name_clean.get(cid, "") for cid in row["cand_ids"]]

        top_matches = process.extract(
            query,
            cand_names,
            scorer=fuzz.WRatio,
            limit=5
        )

        best_score = -1
        best_pos = None

        for name, wr, pos in top_matches:

            pr = fuzz.partial_ratio(query, name)
            ts = fuzz.token_sort_ratio(query, name)
            tset = fuzz.token_set_ratio(query, name)

            combined = max(wr, pr, ts, tset)

            if combined > best_score:
                best_score = combined
                best_pos = pos

        score = best_score

        if score >= threshold:
            matched_ids.append(row["cand_ids"][best_pos])
            scores.append(score)
            loc_dists.append(row["cand_dist_m"][best_pos])
            matched_names.append(id_to_name_raw.get(row["cand_ids"][best_pos]))
            matched_cats.append(id_to_cat.get(row["cand_ids"][best_pos]))
        else:
            matched_ids.append(pd.NA)
            scores.append(score)
            loc_dists.append(pd.NA)
            matched_names.append(pd.NA)
            matched_cats.append(pd.NA)


    result = reference_gdf.copy()
    result["matched_id"] = matched_ids
    result["name_score"] = scores
    result["location_distance"] = loc_dists
    result["matched_name"] = matched_names
    result["matched_cat_main"] = matched_cats

    return result

In [6]:
import pandas as pd
import geopandas as gpd
from rapidfuzz import fuzz
import re
import unicodedata

def clean_name(s):
    if not isinstance(s, str):
        return ""

    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c)) # 1. unicode normalize (remove accents)
    s = s.upper() # 2. uppercase
    s = re.sub(r"\([^)]*\)", "", s) 
    s = s.encode("ascii", errors="ignore").decode() # 4. remove emoji / non ascii
    s = re.sub(r"[^\w\s]", " ", s) # 5. replace special chars with space
    s = re.sub(r"\s+", " ", s) # 6. collapse spaces

    return s.strip()

def address_score_check(
    reference_gdf: gpd.GeoDataFrame,
    compared_gdf: gpd.GeoDataFrame,
    addr_col_ref: str = "addr_simple",
    addr_col_cmp: str = "address",
    matched_id_col: str = "matched_id",
    id_col: str = "id",
    out_col: str = "address_score",
    out_addr_col: str = "matched_address", 
):
    """
    Compute address similarity score (0-100) for already-matched rows.

    Only rows with non-null `matched_id_col` will be scored.
    Others will have NaN.

    Returns
    -------
    GeoDataFrame with a new column `out_col`.
    """

    # map: compared id -> compared address
    id_to_addr_clean = compared_gdf.set_index(id_col)[addr_col_cmp].apply(clean_name).to_dict()
    id_to_addr_raw = compared_gdf.set_index(id_col)[addr_col_cmp].to_dict()

    scores = []
    matched_addrs = []

    for _, row in reference_gdf.iterrows():
        matched_id = row.get(matched_id_col)

        # no matched id -> no score
        if pd.isna(matched_id):
            scores.append(pd.NA)
            matched_addrs.append(pd.NA)
            continue

        addr_ref = clean_name(row.get(addr_col_ref))
        addr_cmp = id_to_addr_clean.get(matched_id)

        if isinstance(addr_cmp, str) and addr_cmp.strip():
            matched_addrs.append(id_to_addr_raw.get(matched_id))
        else:
            matched_addrs.append(pd.NA)

        # missing address on either side -> no score
        if not isinstance(addr_ref, str) or not addr_ref.strip():
            scores.append(pd.NA)
            continue
        if not isinstance(addr_cmp, str) or not addr_cmp.strip():
            scores.append(pd.NA)
            continue

        wr = fuzz.WRatio(addr_ref, addr_cmp)
        pr = fuzz.partial_ratio(addr_ref, addr_cmp)
        ts = fuzz.token_sort_ratio(addr_ref, addr_cmp)
        tset = fuzz.token_set_ratio(addr_ref, addr_cmp)

        scores.append(int(max(wr, pr, ts, tset)))

    result = reference_gdf.copy()
    result[out_col] = scores
    result[out_addr_col] = matched_addrs
    return result

In [7]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

def calculate_similarity_check(
    df, 
    cat_col_ref: str = "primary_type", 
    cat_col_cmp: str = "matched_cat_main", 
    id_col: str = "matched_id", 
    result_col: str =  "category_sim",
):

    # 1. Setup Device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    
    # 2. Primary Gatekeeper: matched_id must be present
    mask = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    
    # 3. Create a helper to handle potential Nulls in text columns
    temp_df = df[mask].copy()
    
    # Identify where text is actually missing within the masked rows
    text_missing_mask = temp_df[cat_col_ref].isna() | temp_df[cat_col_cmp].isna()
    
    # Fill NaNs with empty strings just for the encoding step
    texts_1 = temp_df[cat_col_ref].fillna("").astype(str).tolist()
    texts_2 = temp_df[cat_col_cmp].fillna("").astype(str).tolist()

    print(f"Encoding {len(temp_df)} rows...")

    # 4. Batch Encoding
    emb1 = model.encode(texts_1, batch_size=256, show_progress_bar=True, convert_to_tensor=True)
    emb2 = model.encode(texts_2, batch_size=256, show_progress_bar=True, convert_to_tensor=True)

    # 5. Calculate Similarity
    scores = torch.nn.functional.cosine_similarity(emb1, emb2, dim=1).cpu().numpy()
    
    # 6. Apply NaN to rows where text was missing
    # Even if we encoded an empty string, the result isn't "real" if data was missing
    scores[text_missing_mask.values] = np.nan

    # 7. Map back to original dataframe
    df[result_col] = np.nan
    df.loc[mask, result_col] = scores
    
    return df

In [8]:
chicago_gplc = gpd.read_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_gplc.geojson')
chicago_gplc

,circle_id,id,name,address,primary_type,lat,lon,business_status,primary_cat,addr_simple,naics_code,naics_definition,geometry
0,0,ChIJd1-a836IDYgR6PolD_KgODE,Trudeau's Body Shop,"515 N Clark St, Sheldon, IL 60966, USA",car_repair,40.775277,-87.575841,OPERATIONAL,automotive,515 N Clark St,8111.0,Automotive Repair and Maintenance,POINT (-87.57584 40.77528)
1,0,ChIJRc-9WACJDYgRVZb9tKwxdyc,Village of Sheldon Administration Building Par...,"Sheldon, IL 60966, USA",parking_lot,40.773454,-87.563814,OPERATIONAL,automotive,Sheldon,8123.0,Parking Lots and Garages,POINT (-87.56381 40.77345)
2,1,ChIJh1H7CHN6EogRq3yq0ddXj9k,BP,"409 S 7th St, Kentland, IN 47951, USA",gas_station,40.761758,-87.438293,OPERATIONAL,automotive,409 S 7th St,447.0,Gasoline Stations,POINT (-87.43829 40.76176)
3,1,ChIJ8_MdS55wEogRyr7tM4FNZyA,Marathon Gas,"102 N 7th St, Kentland, IN 47951, USA",gas_station,40.768374,-87.438839,OPERATIONAL,automotive,102 N 7th St,447.0,Gasoline Stations,POINT (-87.43884 40.76837)
4,1,ChIJzaSgXgd6EogRNZrXaY3d98A,Model 1 Chevrolet of Kentland,"308 W Seymour St, Kentland, IN 47951, USA",car_dealer,40.768603,-87.454434,OPERATIONAL,automotive,308 W Seymour St,441.0,Motor Vehicle and Parts Dealers,POINT (-87.45443 40.76860)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56956,428,ChIJJ32HgQ3zD4gRo0W2SYFi0ro,Francis Energy Charging Station,"39131 N Sheridan Rd, Beach Park, IL 60087, USA",electric_vehicle_charging_station,42.429671,-87.825194,OPERATIONAL,transportation,39131 N Sheridan Rd,8111.0,Automotive Repair and Maintenance,POINT (-87.82519 42.42967)
56957,428,ChIJcfhLpBT1D4gRyyucDs7YN8Y,Beach Ridge Day Use Area Parking Lot,"Zion, IL 60099, USA",parking_lot,42.468193,-87.799932,OPERATIONAL,transportation,Zion,8123.0,Parking Lots and Garages,POINT (-87.79993 42.46819)
56958,428,ChIJAbu4aX7zD4gRNF4MA6GYmmw,"Zion,IL Metra Station West Parking Lot","Zion, IL 60099, USA",parking_lot,42.449095,-87.818332,OPERATIONAL,transportation,Zion,8123.0,Parking Lots and Garages,POINT (-87.81833 42.44910)
56959,428,ChIJJ7TMedP1D4gRiLMElMLB6Fg,Bison Pallets,"2415 23rd St, Zion, IL 60099, USA",transportation_service,42.453897,-87.845944,OPERATIONAL,transportation,2415 23rd St,4859.0,Other Transit and Ground Passenger Transportation,POINT (-87.84594 42.45390)


In [9]:
chicago_sf = gpd.read_parquet('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_msa_poi/chicago_sf.parquet')

In [10]:
chicago_gplc_sf = search_spatial_candidates(reference_gdf=chicago_gplc, compared_gdf=chicago_sf, id_col = "PLACEKEY", k=100)

In [11]:
chicago_gplc_sf = match_by_name(reference_gdf=chicago_gplc_sf, compared_gdf=chicago_sf, re_name_col = "name", comp_name_col = "LOCATION_NAME", comp_id = "PLACEKEY",comp_id_col ="TOP_CATEGORY",  threshold=80)

In [12]:
chicago_gplc_sf = address_score_check(reference_gdf=chicago_gplc_sf, compared_gdf=chicago_sf, addr_col_ref = "addr_simple", addr_col_cmp = "STREET_ADDRESS", id_col = "PLACEKEY",)

In [13]:
chicago_gplc_sf = calculate_similarity_check(chicago_gplc_sf, cat_col_ref = "naics_definition", cat_col_cmp = "matched_cat_main", id_col = "matched_id", result_col =  "category_sim")

Encoding 45576 rows...


Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Batches:   0%|          | 0/179 [00:00<?, ?it/s]

In [14]:
chicago_gplc_sf['matched_id'].notnull().sum() / len(chicago_gplc_sf)

0.8001264022752409

In [15]:
# transfer the X and Y into float type and deal with the address score
cols_to_fix = ['name_score', 'location_distance', 'address_score', 'category_sim']
for col in cols_to_fix:
    chicago_gplc_sf[col] = pd.to_numeric(chicago_gplc_sf[col], errors='coerce')

In [ ]:
df = chicago_gplc_sf[chicago_gplc_sf["matched_id"].notna()].copy()

N = 1000

weights = df["primary_cat"].map(
    df["primary_cat"].value_counts(normalize=True)
)

df_sample = df.sample(
    n=N,
    weights=weights,
    random_state=42
)
df_sample[['id','name','addr_simple','naics_definition', 'matched_id','matched_name','matched_address','matched_cat_main','location_distance']].to_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/bsop_gplc_sf_2000_sample.csv', index=False)

In [16]:
df_sample_check = pd.read_csv('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_sf_1000_sample.csv')
df_sample_check = df_sample_check.drop(columns='location_distance')
df_sample_check = df_sample_check.merge(chicago_gplc_sf[['id','name_score','location_distance','address_score','category_sim']], left_on="id", right_on="id", how="left")

In [17]:
df_sample_check.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1000 non-null   object 
 1   name               1000 non-null   object 
 2   addr_simple        1000 non-null   object 
 3   naics_definition   1000 non-null   object 
 4   matched_id         1000 non-null   object 
 5   matched_name       1000 non-null   object 
 6   matched_address    1000 non-null   object 
 7   matched_cat_main   1000 non-null   object 
 8   is_true            1000 non-null   int64  
 9   name_score         1000 non-null   float64
 10  location_distance  1000 non-null   float64
 11  address_score      1000 non-null   float64
 12  category_sim       1000 non-null   float64
dtypes: float64(4), int64(1), object(8)
memory usage: 101.7+ KB


In [18]:
df_sample_check['is_true'].value_counts()

is_true
1    926
0     74
Name: count, dtype: int64

In [19]:
df = df_sample_check.copy()
from sklearn.preprocessing import StandardScaler

X = df[['name_score', 'location_distance', 'address_score', 'category_sim']]
y = df['is_true']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.25,
    random_state=42,
    stratify=y  # keep the same proportion of True vs False in training set and test set
)

In [21]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=False,
    eval_metric='auc',
    random_state=42
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False  
)

xgb_y_pred = xgb_clf.predict(X_test)
xgb_y_prob = xgb_clf.predict_proba(X_test)[:, 1]

xgb_cr = classification_report(y_test, xgb_y_pred)
xgb_auc = roc_auc_score(y_test, xgb_y_prob)

print(xgb_cr)
print("XGBoost AUC:", xgb_auc)

              precision    recall  f1-score   support

           0       0.71      0.89      0.79        19
           1       0.99      0.97      0.98       231

    accuracy                           0.96       250
   macro avg       0.85      0.93      0.89       250
weighted avg       0.97      0.96      0.97       250

XGBoost AUC: 0.9890635680109363


In [22]:
mask = chicago_gplc_sf['matched_id'].notnull()
df_pred = chicago_gplc_sf.loc[mask].copy()

feature_cols = ['name_score', 'location_distance', 'address_score', 'category_sim']

X_new = scaler.transform(df_pred[feature_cols])
df_pred['true_match_prob'] = xgb_clf.predict_proba(X_new)[:, 1]
df_pred['is_true_match'] = df_pred['true_match_prob'] >= 0.5

chicago_gplc_sf.loc[mask, 'is_true_match'] = df_pred['is_true_match']
chicago_gplc_sf.loc[mask, 'true_match_prob'] = df_pred['true_match_prob']

In [23]:
chicago_gplc_sf['is_true_match'].value_counts()

is_true_match
True     40466
False     5110
Name: count, dtype: int64

In [24]:
chicago_gplc_sf.drop(columns=['cand_ids','cand_dist_m']).to_file('/Users/houpuli/Redlining Lab Dropbox/HOUPU LI/POI research/Chicago_MSA/chicago_google_comparison/chicago_gplc_sf.geojson')